In [1]:
import torch
import math

## Clipping a vector and adding Gaussian noise

Gradient clippping:
- Caps the maximum length (L2,norm) of per sample gradients computed for each training example
- It limits sensitivity
  $$\hat{x} = x \cdot \min\left(1, \frac{C}{\|x\|_2}\right)$$
```



```
In DP-SGD the standard deviation of the noise added to each aggregated gradient is propotional to:
$$\sigma_{\text{noise}} = zC$$

- $C$ is the clipping threshold
- $z$ is the nouse multiplier
- $\sigma_{\text{noise}}$ Gaussian noise standard deviation, under the summed-gradient convention
```



```
Formula for the standard deviation of gaussian noise
$$\sigma = \frac{\Delta f \sqrt{2 \ln(1.25/\delta)}}{\epsilon}$$

- $\Delta f$ is the sensitivity
- $\epsilon$ is the primary privacy budget parameter. Smaller ε means stronger privacy, which results in a larger σ (more noise).
- $\delta$ is the the probability of the privacy guarantee failing (information leaking). Smaller δ means stricter security, which also increases the noise.

In [4]:
# update contains a vector of weights and biases
# example update
update = torch.tensor([3.0,4.0])

# calculate L2 norm, measures the size of the vector
norm = torch.linalg.vector_norm(update)

max_norm = 1.0

clipped_update = update * min(1.0, max_norm / norm.item())

print("Original:", update)
print("Original norm:", norm.item())
print("Clipped:", clipped_update)
print("Clipped norm:",
      torch.linalg.vector_norm(clipped_update).item())


Original: tensor([3., 4.])
Original norm: 5.0
Clipped: tensor([0.6000, 0.8000])
Clipped norm: 1.0


In [19]:
torch.manual_seed(123)

delta = 1e-5

# dummy sensitivity
sensitivity = 1.0

# formula for std of gaussian noise
for epsilon in [0.5, 1.0,2.0,5.0]:
    noise_std = (
        sensitivity
        * math.sqrt(2 * math.log(1.25 / delta))
        / epsilon
    )
    
    noise = torch.randn_like(clipped_update) * noise_std
    noisy_update = clipped_update + noise

    print(f"\nepsilon: {epsilon}")
    print("Noise standard deviation:", noise_std)
    print("Clipped update:", clipped_update)
    print("Noise:", noise)
    print("Noisy update:", noisy_update)



epsilon: 0.5
Noise standard deviation: 9.689610525210778
Clipped update: tensor([0.6000, 0.8000])
Noise: tensor([-1.0801,  1.1663])
Noisy update: tensor([-0.4801,  1.9663])

epsilon: 1.0
Noise standard deviation: 4.844805262605389
Clipped update: tensor([0.6000, 0.8000])
Noise: tensor([-1.7908, -1.1648])
Noisy update: tensor([-1.1908, -0.3648])

epsilon: 2.0
Noise standard deviation: 2.4224026313026945
Clipped update: tensor([0.6000, 0.8000])
Noise: tensor([-2.8994,  0.5069])
Noisy update: tensor([-2.2994,  1.3069])

epsilon: 5.0
Noise standard deviation: 0.9689610525210778
Clipped update: tensor([0.6000, 0.8000])
Noise: tensor([-0.9422, -0.7316])
Noisy update: tensor([-0.3422,  0.0684])
